# Module 5: Exploratory Data Analysis — Synthetic Retail Orders

**Question:** What do order patterns suggest about revenue and customer experience? All data are synthetic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
plt.style.use("default")
DATA_PATH = Path("data/retail_orders_synthetic.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("retail_orders_synthetic.csv")  # Colab: upload CSV to Files
df = pd.read_csv(DATA_PATH, parse_dates=["order_date"])
df.head()

## 1. Inspect schema, grain, date range and quality

In [ ]:
print("Shape:", df.shape)
print("Date range:", df.order_date.min(), "to", df.order_date.max())
print("Duplicate IDs:", df.order_id.duplicated().sum())
print(df.dtypes)
quality = pd.DataFrame({"missing_count":df.isna().sum(), "missing_pct":df.isna().mean().mul(100).round(1)})
print(quality.to_string())

## 2. Feature engineering and summary statistics

In [ ]:
df["revenue"] = df.quantity * df.unit_price * (1-df.discount_rate)
print(df[["quantity","unit_price","revenue","delivery_days","customer_satisfaction"]].describe().round(2))
print("Total revenue proxy:",round(df.revenue.sum(),2))
print("Median order revenue:",round(df.revenue.median(),2))

## 3. Revenue distribution and outlier screening

In [ ]:
q1,q3=df.revenue.quantile([.25,.75]); iqr=q3-q1; upper=q3+1.5*iqr
print("IQR upper fence:",round(upper,2),"Flagged orders:",int((df.revenue>upper).sum()))
fig,ax=plt.subplots(figsize=(8,4));ax.hist(df.revenue,bins=30);ax.set(title="Order revenue distribution (synthetic)",xlabel="Revenue proxy");plt.tight_layout();plt.show()

## 4. Category and region comparisons

In [ ]:
category=df.groupby("category",dropna=False).agg(orders=("order_id","count"),total_revenue=("revenue","sum"),median_order=("revenue","median")).sort_values("total_revenue",ascending=False)
print(category.round(2))
fig,ax=plt.subplots(figsize=(8,4));category.total_revenue.plot.bar(ax=ax);ax.set(title="Revenue proxy by category",ylabel="Total revenue proxy");plt.tight_layout();plt.show()

## 5. Channel mix and two-way segmentation

In [ ]:
print(df.channel.value_counts(normalize=True).mul(100).round(1))
seg=df.pivot_table(index="region",columns="channel",values="revenue",aggfunc="mean",observed=True)
print(seg.round(2))
fig,ax=plt.subplots(figsize=(7,4));sns.heatmap(seg,annot=True,fmt=".0f",cmap="Blues",ax=ax);ax.set(title="Mean order revenue by region and channel");plt.tight_layout();plt.show()

## 6. Delivery versus satisfaction: association, not causation

In [ ]:
valid=df[["delivery_days","customer_satisfaction"]].dropna()
print("Pearson correlation:",round(valid.corr().iloc[0,1],3),"n:",len(valid))
fig,ax=plt.subplots(figsize=(7,4));df.boxplot(column="customer_satisfaction",by="delivery_days",ax=ax);ax.set(title="Satisfaction by delivery days");plt.tight_layout();plt.show()

## 7. Time trend

In [ ]:
daily=df.set_index("order_date").resample("W")["revenue"].sum()
fig,ax=plt.subplots(figsize=(9,4));daily.plot(ax=ax,marker="o");ax.set(title="Weekly revenue proxy",ylabel="Revenue proxy");plt.tight_layout();plt.show()

## 8. Numeric correlation matrix

In [ ]:
numeric=df[["quantity","unit_price","discount_rate","revenue","delivery_days","customer_satisfaction"]]
fig,ax=plt.subplots(figsize=(8,5));sns.heatmap(numeric.corr(),annot=True,cmap="coolwarm",center=0,fmt=".2f",ax=ax);plt.tight_layout();plt.show()

## 9. Export summary and write findings
Record 3 findings with exact values from your run, 2 limitations, and 2 next questions. Never call these synthetic results real-world evidence.

In [ ]:
out=Path("outputs");out.mkdir(exist_ok=True)
category.round(2).to_csv(out/"category_summary.csv")
quality.to_csv(out/"data_quality_summary.csv")
print("Saved summaries in",out.resolve())